In [3]:
!pip install scanpy
!pip install AnnData
!pip install numpy
!pip install pandas

In [4]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd

In [5]:
# 下载 PBMC 3k
adata = sc.datasets.pbmc3k()

# 保存为 h5ad 文件
adata.write_h5ad("pbmc3k_raw.h5ad")
print("✅ 已保存 pbmc3k_raw.h5ad")

  0%|          | 0.00/5.58M [00:00<?, ?B/s]

✅ 已保存 pbmc3k_raw.h5ad


In [8]:
adata = ad.read_h5ad("pbmc3k_raw.h5ad")
print(adata)

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)


In [9]:
# ========== 1. 整体概览 ==========
print("=" * 50)
print(adata)
print("=" * 50)

# ========== 2. 细胞级注释 (obs) ==========
print("\n【adata.obs】前 5 行：")
print(adata.obs.head())
print(f"\nobs 列名: {list(adata.obs.columns)}")
print(f"obs 形状: {adata.obs.shape}")

# ========== 3. 基因级注释 (var) ==========
print("\n【adata.var】前 5 行：")
print(adata.var.head())
print(f"\nvar 列名: {list(adata.var.columns)}")

# ========== 4. 多维数组注释 (obsm / varm) ==========
print(f"\n【adata.obsm】keys: {list(adata.obsm.keys())}")
print(f"【adata.varm】keys: {list(adata.varm.keys())}")

# ========== 5. 非结构化注释 (uns) ==========
print(f"\n【adata.uns】keys: {list(adata.uns.keys())}")

# ========== 6. 主矩阵 X 的基本统计 ==========
print(f"\n【adata.X】")
print(f"  类型: {type(adata.X)}")
print(f"  形状: {adata.X.shape}")
print(f"  数据类型: {adata.X.dtype}")
print(f"  稀疏度: {1 - adata.X.nnz / np.prod(adata.X.shape):.2%}")

AnnData object with n_obs × n_vars = 2700 × 32738
    var: 'gene_ids'
    layers: None (.X)

【adata.obs】前 5 行：
Empty DataFrame
Columns: []
Index: [AAACATACAACCAC-1, AAACATTGAGCTAC-1, AAACATTGATCAGC-1, AAACCGTGCTTCCG-1, AAACCGTGTATGCG-1]

obs 列名: []
obs 形状: (2700, 0)

【adata.var】前 5 行：
                     gene_ids
index                        
MIR1302-10    ENSG00000243485
FAM138A       ENSG00000237613
OR4F5         ENSG00000186092
RP11-34P13.7  ENSG00000238009
RP11-34P13.8  ENSG00000239945

var 列名: ['gene_ids']

【adata.obsm】keys: []
【adata.varm】keys: []

【adata.uns】keys: []

【adata.X】
  类型: <class 'scipy.sparse._csr.csr_matrix'>
  形状: (2700, 32738)
  数据类型: float32
  稀疏度: 97.41%


In [10]:
# 质控指标
sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
print("添加 QC 指标后的 obs 列：")
print(list(adata.obs.columns))
# ['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt']

# 标准化 + PCA，产生 obsm
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.tl.pca(adata, n_comps=50)

print(f"\n处理后 obsm keys: {list(adata.obsm.keys())}")
# ['X_pca']
print(f"X_pca 形状: {adata.obsm['X_pca'].shape}")
# (2700, 50)

添加 QC 指标后的 obs 列：
['n_genes_by_counts', 'total_counts']

处理后 obsm keys: ['X_pca']
X_pca 形状: (2700, 50)


In [13]:
# ---------- 按条件筛选（如 total_counts 最高的 100 个）----------
top100_idx = adata.obs['total_counts'].nlargest(100).index
adata_100 = adata[top100_idx].copy()

print(f"筛选后: {adata_100}")
print(f"  n_obs = {adata_100.n_obs}")   # 100
print(f"  n_vars = {adata_100.n_vars}") # 32738（基因数不变）
print(f"  obsm keys = {list(adata_100.obsm.keys())}")  # X_pca 自动跟随
print(f"  X_pca shape = {adata_100.obsm['X_pca'].shape}")  # (100, 50) ✅

筛选后: AnnData object with n_obs × n_vars = 100 × 32738
    obs: 'n_genes_by_counts', 'total_counts'
    var: 'gene_ids', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'log1p', 'hvg', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: None (.X)
  n_obs = 100
  n_vars = 32738
  obsm keys = ['X_pca']
  X_pca shape = (100, 50)


In [14]:
output_path = "pbmc3k_100cells.h5ad"
adata_100.write_h5ad(output_path)
print(f"✅ 已保存: {output_path}")

# 验证：重新读取
adata_check = ad.read_h5ad(output_path)
print(f"验证读取: {adata_check}")
# AnnData object with n_obs × n_vars = 100 × 32738

# 查看文件大小
import os
size_mb = os.path.getsize(output_path) / 1024 / 1024
print(f"文件大小: {size_mb:.2f} MB")
# 约 1~2 MB

✅ 已保存: pbmc3k_100cells.h5ad
验证读取: AnnData object with n_obs × n_vars = 100 × 32738
    obs: 'n_genes_by_counts', 'total_counts'
    var: 'gene_ids', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: None (.X)
文件大小: 18.12 MB
